In [7]:
import os
import sys
import re
import pandas as pd


## Just need to set these four variables
file_path = r"/Users/sumitsoni/Documents/Vscode/dummy_data_for_validation/Failure_Tagging_Output - Updated_CR_Refresh Sheet_23rd Feb (3).csv"
master_schema_path = r"/Users/sumitsoni/Documents/Vscode/My_Scripts/Data/Boeing Datatables schema mastersheet.xlsx"
dataset_sheet = "CR"
refresh_type = "Tagged"  
output_file = "datatype_errors.xlsx" # rename it as per your conveinence 


warnings_log = []


# HELPERS fucntions
def kill_script(msg):
    print("\nSCRIPT TERMINATED")
    print(msg)

    if warnings_log:
        print("\nWARNINGS SUMMARY")
        for w in warnings_log:
            print(f"- {w}")

    sys.exit(1)


def print_df_summary(df, title):
    print(f"\n{title}")
    print(f"Total Rows   : {df.shape[0]}")
    print(f"Total Columns: {df.shape[1]}")
    print("\nNull Values Per Column:")
    print(df.isna().sum())

# this needs to be checked


def to_datetime_two_formats(series):
    s = (
        series.astype(str)
        .replace({"nan": "", "NaT": ""})
        .str.replace("\u00A0", "", regex=False)   # NBSP
        .str.replace("\t", "", regex=False)       # tab
        .str.replace("\r", "", regex=False)       # carriage return
        .str.replace("\n", "", regex=False)       # newline
        .str.strip()
    )

    s = s.replace("", pd.NA)

    # Fix: 2023-04-22 0:00:00 -> 2023-04-22 00:00:00
    s = s.str.replace(r"(\d{4}-\d{2}-\d{2})\s(\d{1}):", r"\1 0\2:", regex=True)

    dt1 = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    dt2 = pd.to_datetime(s, format="%Y-%m-%d %H:%M:%S", errors="coerce")

    return dt1.fillna(dt2)


## make it just a warning/ error.
def starts_with_bad_symbol(val):
    if val is None:
        return False
    s = str(val).strip()
    if s == "" or s.lower() in ["nan", "nat"]:
        return False
    return s[0] in [",", ";", "$" ,":", "|","."]


def is_valid_delimited_list(val, delimiter):
    if val is None:
        return True
    s = str(val).strip()
    if s == "" or s.lower() in ["nan", "nat"]:
        return True

    if s.startswith(delimiter):
        return False
    if s.endswith(delimiter):
        return False
    if delimiter * 2 in s:  ## cases like $$ or ,,
        return False
    if starts_with_bad_symbol(s):
        return False

    return True

## for cases like ["Captain's hooks "] ,

def is_valid_list_quote_aware(val):

    if val is None:
        return True

    s = str(val).strip()

    if s == "" or s.lower() in ["nan", "nat"]:
        return True

    # must start and end with list brackets
    if not (s.startswith("[") and s.endswith("]")):
        return False

    inside_quote = False
    quote_char = None
    i = 1  # skip '['

    while i < len(s) - 1:  # skip final ']'
        ch = s[i]

        # 🚩 Flag ANY escape character immediately
        if ch == "\\":
            return False

        # --- Handle quote toggling ---
        if ch in ["'", '"']:
            if not inside_quote:
                inside_quote = True
                quote_char = ch
            elif ch == quote_char:
                inside_quote = False
                quote_char = None

                # after closing quote, next char must be comma, ] or space
                j = i + 1
                while j < len(s) - 1 and s[j] == " ":
                    j += 1

                if j < len(s) - 1 and s[j] not in [",", "]"]:
                    return False

        else:
            if not inside_quote:
                if ch not in [",", " "]:
                    return False

        i += 1

    # valid only if we ended outside a quote
    return not inside_quote

ext = os.path.splitext(file_path)[1].lower()

if ext == ".csv":
    df = pd.read_csv(file_path, low_memory=False)
elif ext in [".xlsx", ".xls"]:
    df = pd.read_excel(file_path)
else:
    kill_script("Unsupported file type. Only CSV or Excel allowed.")

df.columns = df.columns.astype(str).str.strip()
input_columns = df.columns.tolist()


schema_df = pd.read_excel(master_schema_path, sheet_name=dataset_sheet)
schema_df.columns = schema_df.columns.astype(str).str.strip()

required_cols = ["Column Name", "Datatype", "Refresh type"]
for col in required_cols:
    if col not in schema_df.columns:
        kill_script(f"Master sheet '{dataset_sheet}' missing required column: {col}")

schema_filtered = schema_df[
    schema_df["Refresh type"].astype(str).str.strip().str.lower() == refresh_type.lower()
]

if schema_filtered.empty: # just a extra precaution
    kill_script(f"No schema rows found for Refresh type = {refresh_type}")


master_columns = (
    schema_filtered["Column Name"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

datatype_map = (
    schema_filtered.set_index("Column Name")["Datatype"]
    .dropna()
    .astype(str)
    .str.strip()
    .to_dict()
)


# -----------------------------
# COLUMN VALIDATION
# -----------------------------
master_set = set(master_columns)
input_set = set(input_columns)

extra_cols = sorted(list(input_set - master_set))
missing_cols = sorted(list(master_set - input_set))

if extra_cols:
    print(f"columns not in schema: {extra_cols}")

    kill_script(f"Extra columns found in input file that are not present in master schema sheet: {extra_cols}")

if missing_cols:
    warnings_log.append(f"Missing columns in refresh file (allowed): {missing_cols}")


datatype_errors = []

for col in input_columns:

    expected_dtype = datatype_map.get(col, "").strip()

    if expected_dtype == "":
        warnings_log.append(f"Datatype missing in master schema for column: {col}")
        continue

    series = df[col]
    cleaned = (
        series.astype(str)
        .str.strip()
        .replace({"": None, "nan": None, "NaT": None})
    )

    # DATE
    if expected_dtype.upper() == "DATE":
        converted = to_datetime_two_formats(df[col])

        invalid_mask = df[col].notna() & converted.isna()

        if invalid_mask.any():
            bad_rows = df.loc[invalid_mask].copy()
            bad_rows["Failed_Column"] = col
            bad_rows["Expected_Datatype"] = expected_dtype
            datatype_errors.append(bad_rows)

    # NUMBER
    elif expected_dtype.upper() == "NUMBER":

        converted = pd.to_numeric(cleaned, errors="coerce")
        invalid_mask = cleaned.notna() & converted.isna()

    # STRING / FREE TEXT
    elif expected_dtype.upper() in ["STRING", "FREE TEXT"]:

        invalid_mask = cleaned.notna() & cleaned.apply(starts_with_bad_symbol)

    # LIST - ARRAY
    elif expected_dtype.upper() == "LIST - ARRAY":

        invalid_mask = series.notna() & ~series.astype(str).apply(is_valid_list_quote_aware)

    # LIST - DOLLAR SEPARATED
    elif expected_dtype.upper() == "LIST - DOLLAR SEPARATED":

        invalid_mask = series.notna() & ~series.astype(str).apply(
            lambda x: is_valid_delimited_list(x, "$")
        )

    # LIST - COMMA SEPARATED
    elif expected_dtype.upper() == "LIST - COMMA SEPARATED":

        invalid_mask = series.notna() & ~series.astype(str).apply(
            lambda x: is_valid_delimited_list(x, ",")
        )

    else:
        warnings_log.append(f"Unknown datatype '{expected_dtype}' for column: {col}")
        continue

    if invalid_mask.any():
        bad_rows = df.loc[invalid_mask].copy()
        bad_rows["Failed_Column"] = col
        bad_rows["Expected_Datatype"] = expected_dtype
        datatype_errors.append(bad_rows)


if datatype_errors:

    error_df = pd.concat(datatype_errors, ignore_index=True)

    warnings_log.append("Datatype errors are there")
    print("Data has issues\n")
    print("Total rows with datatype issues:", len(error_df))
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        for dtype in error_df["Expected_Datatype"].unique():

            sheet_df = error_df[
                error_df["Expected_Datatype"] == dtype
            ].copy()

            # Excel sheet names cannot exceed 31 characters
            sheet_name = str(dtype)[:31]

            sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)

    print(f"\nError file generated: {output_file}")


else :
    print("\n \n there are no datatype errors \n \n ")


print_df_summary(df, "INPUT FILE SUMMARY")

if warnings_log:
    for w in warnings_log:
        print(f"- {w}")



Data has issues

Total rows with datatype issues: 3

Error file generated: datatype_errors.xlsx

INPUT FILE SUMMARY
Total Rows   : 2883
Total Columns: 3

Null Values Per Column:
CR Id                            0
CLEAN_L2_Failure Condition_CR    0
CLEAN_Failure Component_CR       0
dtype: int64
- Missing columns in refresh file (allowed): ['1_Year_Removal', '90_Days_Removal', 'AC_Equipment_Hours', 'AC_Removals', 'AC_Rolling_MTBUR', 'Air Fleet Average Service Days', 'Airline Cycle Count', 'Airline Cycle_12M_Count', 'Airline Cycle_3M_Count', 'Airline Cycle_6M_Count', 'Average Service Days', 'CLEAN_Failure Condition_CR', 'CLEAN_Fault Code_CR', 'CLEAN_Full_Component_CR', 'CLEAN_L1_Failure Component_CR', 'CLEAN_L1_Maintenance Component_CR', 'CLEAN_L1_Unique_Component_CR', 'CLEAN_L2_Failure Component_CR', 'CLEAN_L2_Maintenance Component_CR', 'CLEAN_L2_Unique_Component_CR', 'CLEAN_Location_Unique_CR', 'CLEAN_MEL Code_CR', 'CLEAN_Maintenance Action_CR', 'CLEAN_Maintenance Component_CR', 'CLEAN

,CR Id,CLEAN_L2_Failure Condition_CR,CLEAN_Failure Component_CR,Failed_Column,Expected_Datatype
0,182139673,"['Not Mentioned','Not Illuminated']",['Cabin Pressure Control Panel CPC CONT PNL AS...,CLEAN_Failure Component_CR,LIST - Array
1,86387886,['Unreliable'],['Audio Control Panel CAP ACP \'],CLEAN_Failure Component_CR,LIST - Array
2,88219638,['Not ILLUM'],['Audio Control Panel ACP L \'],CLEAN_Failure Component_CR,LIST - Array


Row 267 failed:
['Cabin Pressure Control Panel CPC CONT PNL ASSY','Cabin Pressure Control Panel FWD OFV \']
Error: unterminated string literal (detected at line 1) (<unknown>, line 1)
--------------------------------------------------
Row 1531 failed:
['Audio Control Panel CAP ACP \']
Error: unterminated string literal (detected at line 1) (<unknown>, line 1)
--------------------------------------------------
Row 1951 failed:
['Audio Control Panel ACP L \']
Error: unterminated string literal (detected at line 1) (<unknown>, line 1)
--------------------------------------------------
